# Task 1-2: Dataset audit and pose-estimation benchmark

This notebook completes the first two capstone tasks using the **real squat videos and error annotations already present in this repository**.

It does two things:
1. Audits the local data to determine which labels already exist and whether the project can support exercise-type and quality-label learning.
2. Benchmarks two real pose-estimation models on the local videos: **MediaPipe BlazePose** and **YOLO11n-pose**.

The notebook also records the gap between what is already available locally and what will still be needed later for full multi-exercise recognition.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, Markdown, display

from scripts.task1_task2_pose_benchmark import run_pipeline

outputs = run_pipeline(total_benchmark_videos=8, max_frames=75, confidence_threshold=0.5)
project_root = Path.cwd()
reports_dir = project_root / 'reports'
figures_dir = project_root / 'figures'

manifest = outputs['manifest']
dataset_options = outputs['dataset_options']
benchmark_videos = outputs['benchmark_videos']
benchmark_detail = outputs['benchmark_detail']
benchmark_summary = outputs['benchmark_summary']
summary = outputs['summary']

with (reports_dir / 'task1_dataset_summary.json').open('r', encoding='utf-8') as handle:
    summary_json = json.load(handle)

print('Manifest rows:', len(manifest))
print('Benchmark sample videos:', len(benchmark_videos))
print('Recommended model:', benchmark_summary.iloc[0]['model_name'])

## Task 1: Which datasets are suitable?

### What the repository already gives us
- Every local video in `videos_squat/` is a **squat** clip, so the exercise label is present but only for a **single class**.
- The project also includes timestamped quality annotations in `error_knees_forward.json` and `error_knees_inward.json`.
- That means the repo is already useful for a **squat-form assessment pilot**, but not yet for the later multi-exercise classifier from the capstone summary.

### Public datasets worth noting
- **Fitness-AQA** is the closest paper-aligned option because it includes multiple gym exercises plus quality labels.
- **UI-PRMD** is a weaker but still useful backup because it contains multiple movement types plus correct/incorrect performance examples.

### Honest conclusion for this project today
Use the repository data as the **primary real dataset for squat-quality work now**, and treat Fitness-AQA / UI-PRMD as expansion options for later stages that need more exercise classes.

In [ ]:
summary_table = pd.DataFrame([
    {
        'metric': 'total_videos',
        'value': summary_json['total_videos'],
    },
    {
        'metric': 'exercise_types_present',
        'value': ', '.join(summary_json['exercise_types_present']),
    },
    {
        'metric': 'supports_multiple_exercise_types',
        'value': summary_json['task_1_assessment']['supports_multiple_exercise_types'],
    },
    {
        'metric': 'mean_duration_seconds',
        'value': summary_json['mean_duration_seconds'],
    },
    {
        'metric': 'median_duration_seconds',
        'value': summary_json['median_duration_seconds'],
    },
])

display(summary_table)
display(Markdown('### Split counts'))
display(pd.DataFrame(summary_json['split_counts'].items(), columns=['split', 'videos']))
display(Markdown('### Quality-label counts derived from the repo annotations'))
display(pd.DataFrame(summary_json['quality_label_counts'].items(), columns=['quality_label', 'videos']).sort_values('videos', ascending=False))
display(Markdown('### Candidate datasets'))
display(dataset_options)

## Task 2: Pose-estimation benchmark on local squat videos

The benchmark uses a stratified sample of repository videos and compares two models that are practical for this capstone:
- **MediaPipe BlazePose** for fast CPU inference and stable landmark tracking.
- **YOLO11n-pose** for detector-style 17-keypoint prediction.

Because the repository does **not** include ground-truth joint coordinates, the comparison uses the strongest accuracy proxies that can be measured on the real project data:
- visible-keypoint ratio,
- fraction of fully tracked frames,
- average keypoint confidence,
- temporal jitter proxy,
- effective inference FPS.

This is not a substitute for a full keypoint-ground-truth benchmark, but it is enough to choose the more practical extractor for downstream feature engineering.

In [ ]:
display(Markdown('### Benchmark video sample'))
display(benchmark_videos[['video_id', 'split', 'quality_label', 'frame_count', 'duration_seconds']])

display(Markdown('### Model-level benchmark summary'))
display(benchmark_summary)

display(Markdown('### Per-video benchmark detail (first rows)'))
display(benchmark_detail.head(12))

recommended = benchmark_summary.iloc[0]
display(Markdown(
    f"**Recommended model for this project:** {recommended['model_name']}  "
    f"(selection score = {recommended['selection_score']:.3f}, "
    f"mean FPS = {recommended['mean_effective_fps']:.2f}, "
    f"visible-keypoint ratio = {recommended['mean_visible_keypoint_ratio']:.3f})"
))

In [ ]:
display(Markdown('### Overlay comparison on one repository video'))
display(Image(filename=str(figures_dir / 'task2_pose_overlay_comparison.png')))

display(Markdown(
    'BlazePose is expected to be the stronger starting point here if it wins on both speed and tracking coverage, '
    'because downstream rep counting and angle-based quality scoring need dense, stable trajectories more than detector-style sparsity.'
))

## Outcome

- **Task 1 is completed for the current repository scope**: the notebook converts the raw project assets into a usable dataset manifest and states clearly that the local data supports **squat-quality learning** but not yet **multi-exercise classification**.
- **Task 2 is completed for the current repository scope**: both pose models are run on the real project videos and compared with reproducible outputs saved to `reports/` and `figures/`.
- The selected pose model is the best next step for tasks 3-6, where the project will normalise keypoints, compute joint angles, and train sequence models.